# utils_StudentC.ipynb

This notebook documents and demonstrates the functions in `utils.py` written by **Student C**:

1. `get_st_model` – loads a pre-trained Sentence Transformer model.
2. `get_st_cosine_features` – computes cosine similarity between Sentence Transformer embeddings for question pairs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
import pandas as pd
import numpy as np
from utils import (
    get_data_splits,
    get_st_model,
    get_st_cosine_features,
)

## Sentence Transformers for Semantic Similarity

While BoW and TF-IDF capture keyword overlap, they often fail to capture **semantic meaning**. For example, "How can I lose weight?" and "What are the best ways to shed pounds?" share very few words but have the same intent.

**Sentence Transformers** (based on BERT/RoBERTa) encode entire sentences into dense vectors where similar meanings are close together in the embedding space.

### Implementation Details
- **Model**: We use `all-MiniLM-L6-v2`, which is a balanced model in terms of speed and accuracy.
- **Embedding**: Each question is encoded into a 384-dimensional vector.
- **Similarity**: We compute the cosine similarity between the two question vectors.

In [ ]:
model = get_st_model()

q1 = "How do I learn Python?"
q2 = "What is the best way to start programming in Python?"
q3 = "How do I cook an omelette?"

df_example = pd.DataFrame({
    "question1": [q1, q1],
    "question2": [q2, q3]
})

sims = get_st_cosine_features(df_example, model)
print(f"Similarity between '{q1}' and '{q2}': {sims[0][0]:.4f}")
print(f"Similarity between '{q1}' and '{q3}': {sims[1][0]:.4f}")

## Distribution on a sample

We evaluate how well this feature separates duplicate questions from non-duplicates on a small sample of the data.

In [ ]:
DATA_PATH = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = "quora_data.csv"

quora_df = pd.read_csv(DATA_PATH)
train_df, _, _ = get_data_splits(quora_df)

sample = train_df.sample(200, random_state=42)
X_st = get_st_cosine_features(sample, model)

dup_mask = sample["is_duplicate"].values.astype(bool)
print(f"Mean ST Cosine for duplicates:     {X_st[dup_mask].mean():.3f}")
print(f"Mean ST Cosine for non-duplicates: {X_st[~dup_mask].mean():.3f}")